# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, following best practices for referencing data entities by their `@id` fields.

### Dataset Source
The dataset Croissant schema is available from:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and browse the records in the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Let's review the available record sets, their fields, and all their associated `@id` values.

**Note:** All Croissant entities are referenced by their `@id` field!

In [ ]:
# List all record sets including their @id and fields
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for idx, record_set in enumerate(record_sets, 1):
    print(f"Record Set {idx}: @id = {record_set.id}")
    print(f"           Name: {getattr(record_set, 'name', getattr(record_set, 'id', ''))}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, name: {getattr(field, 'name', field.id)}")
    if hasattr(record_set, 'columns') and record_set.columns:
        print(f"  Columns:")
        for col in record_set.columns:
            print(f"    - @id: {col.id}, name: {getattr(col, 'name', col.id)}")
    print()

## 3. Data Extraction
Extract data from a record set using its `@id` and explore the available fields.

Below, we load each record set as a DataFrame using the `@id` for consistent referencing.

In [ ]:
# Get available record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids:", record_set_ids)

# Load all record sets into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample:")
    print(df.head(2))

# For further processing, choose the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[main_record_set_id] if main_record_set_id else None
print(f"Main record set @id: {main_record_set_id}")
if main_df is not None:
    print("Columns in the main record set:", main_df.columns.tolist())
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's analyze numeric fields, filter records, normalize values, and group by relevant fields.

**Tip:** Use the field `@id` (not just the column name) to reference columns corresponding to the Croissant schema IDs.

In [ ]:
if main_df is not None:
    # Try to infer which columns are numeric for demonstration
    numeric_cols = main_df.select_dtypes(include='number').columns.tolist()
    print("Numeric fields detected (by @id):", numeric_cols)
    
    # Use the first numeric field, or prompt the user to specify
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = main_df[numeric_field_id].quantile(0.9)
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 10%): {len(filtered_df)} records\n")

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )

        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a non-numeric field
        non_num_cols = [c for c in main_df.columns if c not in numeric_cols]
        group_field_id = non_num_cols[0] if non_num_cols else None
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id} (showing top groups):")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No records available to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the selected record set using plotly or matplotlib.

In [ ]:
import matplotlib.pyplot as plt

if main_df is not None and numeric_cols:
    plt.figure(figsize=(8, 5))
    main_df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id} (@id reference)')
    plt.show()
    
    # If a group field was used, show mean by group
    if group_field_id:
        group_means = main_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        group_means.plot(kind='bar', figsize=(10, 4))
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} for each {group_field_id} (@id)')
        plt.show()

## 6. Conclusion
- We demonstrated how to load, browse, and analyze records from the FAIR^2 dataset Croissant package, referencing all record sets, fields, and columns via their precise `@id` in the schema.
- You can extend this notebook by further analyzing other record sets or by exporting processed data for downstream analysis.

To learn more about the [mlcroissant](https://mlcommons.github.io/croissant/) library and the [FAIR^2 dataset](https://doi.org/10.71728/senscience.vq4a-28xa), refer to their respective documentation.